# TVA plant scheduling: starter (project version)

Same flow as `reference/upstream/starter.ipynb`, but the code lives in `src/` so tests and scripts share it.
**Fill in the TODOs in `src/`, not here** — autoreload picks up your edits.

| Section | Milestone | Code | TODO |
|---|---|---|---|
| 1 | 1 | `src/config.py`, `src/check.py` | hand schedule (below) |
| 2 | 2 | `src/classical.py` | **TODO 1** `cheapest_first` |
| 3 | 3 | `src/qubo.py` | **TODO 2** `plant_hour_costs`, **TODO 3** `build_qubo` |
| 4 | 4 | `src/quantum.py` | — |
| 5–6 | 4, 6 | `src/scheduler.py` | — |
| 7 | 5 | `src/hardware.py` | — |

Check your TODOs any time with `pytest` from the repo root.

In [ ]:
# Pick the qBraid-SDK (v0.12.2) kernel in qBraid Lab. Locally, any env with requirements.txt works.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))

try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

%load_ext autoreload
%autoreload 2

import time
import numpy as np
import matplotlib.pyplot as plt

from src.check import check_schedule, check_hour, hour_ok, locks
from src.config import COLORS, N_SLACK, load_problem
from src.classical import cheapest_first, run_milp
from src.qubo import (plant_hour_costs, build_qubo, brute_force_qubo, label, choice_to_index, index_to_choice)
from src import plotting

FLEET = "5"                      # "10" for the stretch goal
problem = load_problem(FLEET)
plants, demand, reserve, N, names, hours = (problem.plants, problem.demand, problem.reserve,
                                            problem.n, problem.names, problem.hours)
print(f"{N} plants, {len(demand)} hours")

## 1. The plants and the demand (Milestone 1)

In [ ]:
print(f"{'plant':<10}{'type':<32}{'pmin':>6}{'pmax':>6}{'$/MWh':>8}{'no-load $/h':>13}"
      f"{'startup $':>11}{'min up':>8}{'min down':>10}")
for p in plants:
    print(f"{p['plant_id']:<10}{p['type']:<32}{p['pmin_mw']:>6.0f}{p['pmax_mw']:>6.0f}"
          f"{p['marginal_cost_per_mwh']:>8.2f}{p['no_load_cost_per_h']:>13,.0f}"
          f"{p['startup_cost']:>11,.0f}{p['min_up_h']:>7.0f}h{p['min_down_h']:>9.0f}h")
plotting.plot_demand(problem)
plt.show()

### Milestone 1: schedule 6 hours by hand
Edit `my_schedule`: one row per hour (hours 0 to 5), one column per plant, 1 = on.

In [ ]:
#               COAL  GAS_CC  PEAKER_A  PEAKER_B  OIL
my_schedule = [[1,    0,      0,        0,        0],   # hour 0
               [1,    0,      0,        0,        0],   # hour 1
               [1,    0,      0,        0,        0],   # hour 2
               [1,    0,      0,        0,        0],   # hour 3
               [1,    0,      0,        0,        0],   # hour 4
               [1,    0,      0,        0,        0]]   # hour 5

hand_result = check_schedule(plants, demand[:6], reserve[:6], my_schedule)

## 2. Classical baseline (Milestone 2)

The MILP is `solve_milp` in `src/check.py` — read it once. **TODO 1** is `cheapest_first` in `src/classical.py`.

In [ ]:
milp = run_milp(problem)
milp_cost, milp_schedule = milp["cost"], milp["schedule"]
print(f"MILP solved in {milp['seconds']:.3f} s")
milp_result = check_schedule(plants, demand, reserve, milp_schedule, milp_cost=milp_cost)
plotting.plot_schedule(milp_schedule, names, "MILP schedule")
plt.show()

In [ ]:
rule_schedule = cheapest_first(plants, demand, reserve)          # TODO 1 in src/classical.py
rule_result = check_schedule(plants, demand, reserve, rule_schedule, milp_cost=milp_cost)
plotting.plot_schedule(rule_schedule, names, "Cheapest-first rule")
plt.show()

## 3. One hour as a QUBO (Milestone 3)

$$E = \sum_i c_i x_i + A \Big(\sum_i a_i x_i - (s_0 + 2 s_1 + 4 s_2) - r\Big)^2$$

**TODO 2** (`plant_hour_costs`) and **TODO 3** (`build_qubo`) are in `src/qubo.py`; the docstrings carry the formulas and hints from the upstream notebook.
Done when the cell below prints **Match!** — and you can say in one sentence how the primer's Max-Cut cost relates to yours.

Questions from upstream to answer in the deck: find an hour and a choice that (a) has more than 7 spare blocks and gets penalized though allowed, and (b) breaks "minimum outputs can't exceed demand". Which step in section 5 catches them?

In [ ]:
TEST_HOUR = 12
was_on = milp_schedule[TEST_HOUR - 1]        # who was on the hour before (from the MILP)
Q, offset = build_qubo(plants, demand[TEST_HOUR], reserve[TEST_HOUR], was_on)
bits, energy = brute_force_qubo(Q, offset)
print(f"Cheapest QUBO bitstring: plants {bits[:N]}, slack {bits[N:]}, energy {energy:,.0f}")
check_hour(plants, demand[TEST_HOUR], reserve[TEST_HOUR], bits[:N], was_on=was_on)
print("(The MILP picked", label(list(milp_schedule[TEST_HOUR]), names), "for this hour.)")

In [ ]:
# Every hour at once, with the MILP's previous hour as "who was on".  (Script version: python -m experiments.hour_check)
matches = 0
for t in hours:
    was = milp_schedule[t - 1] if t > 0 else problem.start_state
    Qt, off_t = build_qubo(plants, demand[t], reserve[t], was)
    bits_t, _ = brute_force_qubo(Qt, off_t)
    matches += check_hour(plants, demand[t], reserve[t], bits_t[:N], was_on=was, verbose=False)["matches_best"]
print(f"Your QUBO's cheapest answer matches the brute-force best in {matches} of {len(hours)} hours.")

## 4. QAOA for one hour

Provided in `src/quantum.py`. Remember: `run_qaoa` scores all bitstrings to tune the angles — count that classical work in your timing.

In [ ]:
from src.quantum import run_qaoa, choice_probs, qaoa_circuit

start = time.perf_counter()
angles, probs = run_qaoa(Q, offset, p=1)
print(f"QAOA for hour {TEST_HOUR}: {time.perf_counter() - start:.2f} s, angles {np.round(angles, 3)}")

pc = choice_probs(probs, N)
best_k = choice_to_index(bits[:N])
is_valid = np.array([hour_ok(plants, demand[TEST_HOUR], reserve[TEST_HOUR], index_to_choice(k, N)) for k in range(2 ** N)])
plotting.plot_choice_probs(pc, names, best_k, is_valid, f"QAOA, hour {TEST_HOUR}: most likely on/off choices (orange = best)")
plt.show()
print(f"P(best choice) = {pc[best_k]:.3f}   random guess = {1 / 2 ** N:.3f}")

## 5. The whole day, hour by hour (Milestone 4)

`schedule_day` in `src/scheduler.py`: locks → QUBO + lock penalty → QAOA (warm start) → sample → keep cheapest valid → carry forward.

In [ ]:
from src.scheduler import schedule_day

start = time.perf_counter()
qaoa_schedule, qaoa_log = schedule_day(problem, p=1, shots=200)
print(f"QAOA for 24 hours: {time.perf_counter() - start:.1f} s\n")
qaoa_result = check_schedule(plants, demand, reserve, qaoa_schedule, milp_cost=milp_cost)

print(f"\n{'hour':>4}  {'choice':<28}{'best for this hour?':>20}{'P(best)':>9}{'valid samples':>15}")
for row in qaoa_log:
    print(f"{row['hour']:>4}  {label(row['pick'], names):<28}{'yes' if row['matches_best'] else 'no':>20}"
          f"{row['p_best']:>9.3f}{row['valid_share']:>14.0%}")

## 6. Results vs the baseline

**Required for milestone 4:** QAOA vs random at 5, 10, 20, 50, 200 shots, ≥ 5 seeds each — run
`python -m experiments.shots_sweep` (writes `results/shots_sweep.csv`). Explain the table before tuning anything.

In [ ]:
random_schedule, random_log = schedule_day(problem, sampler="random")
random_result = check_schedule(plants, demand, reserve, random_schedule, milp_cost=milp_cost, verbose=False)

results = {"MILP (optimal)": (milp_result, COLORS["milp"]),
           "cheapest-first rule": (rule_result, COLORS["rule"]),
           "QAOA, hour by hour": (qaoa_result, COLORS["qaoa"]),
           "random guesses, hour by hour": (random_result, COLORS["random"])}
print(f"{'method':<30}{'daily cost $':>14}{'gap':>10}{'rules broken':>14}")
for name, (r, _) in results.items():
    print(f"{name:<30}{r['total_cost']:>14,.0f}{r['gap_pct']:>+9.2f}%{len(r['violations']):>14}")

plotting.plot_comparison(results, qaoa_log, random_log)
plt.show()
fig, axes = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
plotting.plot_schedule(milp_schedule, names, "MILP", ax=axes[0]).set_xlabel("")
plotting.plot_schedule(qaoa_schedule, names, "QAOA, hour by hour", ax=axes[1])
plt.tight_layout()
plt.show()

## 7. One hour on IonQ hardware (Milestone 5)

Order: Aer → IonQ simulator with `forte-1` noise → QPU (final circuit, `p=1`, ≤ 200 shots, send early).
Script version: `python -m experiments.hardware_run` (logs every job to `results/hardware_runs.csv`).

In [ ]:
from src.hardware import build_hardware_circuit, run_aer, submit_qbraid, log_run, choice_shares, device_kind
from src.config import DEVICES

HW_HOUR = 14                 # near the peak (hour 15 is the peak); try others
entry = qaoa_log[HW_HOUR]
hw_angles, hw_probs = run_qaoa(entry["Q"], entry["offset"], p=1, start=entry["angles"])
qc, cx = build_hardware_circuit(entry["Q"], hw_angles, p=1)
print(f"Hour {HW_HOUR}: {qc.num_qubits} qubits, {cx} CX gates after transpiling")
sim_counts = run_aer(qc, shots=200)

In [ ]:
# 1) Free: IonQ simulator with the forte-1 noise model
device_id = DEVICES["ionq-sim"]
job = submit_qbraid(qc, device_id, shots=200, noise_model="forte-1")

# 2) Real hardware (credits!) — only your final, tuned circuit:
# device_id = DEVICES["ionq-qpu"]          # or DEVICES["ionq-qpu-aws"] if unavailable
# job = submit_qbraid(qc, device_id, shots=200, allow_qpu=True)

# 3) Free second opinion (label as EMULATOR): accept Quantinuum's terms in qBraid first
# device_id = DEVICES["quantinuum-emu"]
# job = submit_qbraid(qc, device_id, shots=200)

print("Device:", device_id, "| Job ID:", job.id)
log_run(device_id, job.id, 200, HW_HOUR, 1, cx, status="submitted")
counts = job.result().data.get_counts()
log_run(device_id, job.id, 200, HW_HOUR, 1, cx, status="completed", counts=counts)

In [ ]:
must_on, must_off = entry["must_on"], entry["must_off"]
hw = check_hour(plants, demand[HW_HOUR], reserve[HW_HOUR], qaoa_schedule[HW_HOUR],
                was_on=entry["was_on"], must_on=must_on, must_off=must_off, verbose=False)
if hw["best_choice"] is None:
    raise ValueError(f"No valid on/off choice at hour {HW_HOUR}; pick another HW_HOUR")
best_k = choice_to_index(hw["best_choice"])
is_valid = np.array([hour_ok(plants, demand[HW_HOUR], reserve[HW_HOUR], index_to_choice(k, N), must_on, must_off)
                     for k in range(2 ** N)])
sim_share, hw_share = choice_shares(sim_counts, N), choice_shares(counts, N)
hw_label = f"{device_id} ({device_kind(device_id)})"
for name, share in [("simulator (ideal)", sim_share), (hw_label, hw_share)]:
    print(f"{name:<50} valid shots {share[is_valid].sum():5.0%}   best choice {share[best_k]:5.1%}")
plotting.plot_hardware_vs_sim(sim_share, hw_share, hw_label, names, best_k, is_valid, HW_HOUR)
plt.show()

## What to have ready for the judges

See `docs/results.md` (fill-in tables) and `presentation/demo-notes.md` (five slides).